In [12]:
import pdfplumber
import pandas as pd
import re


In [13]:
#READ PDF FILES
def read_pdf(file_path):
    text = []
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text.append(content)
    return "\n".join(text)


In [14]:
#SPLIT THE SESSIONS INSIDE EACH PDF
def split_sessions(text):
    sessions = re.split(r"Session\s+\d+", text)
    return sessions[1:]  # skip intro


In [15]:
#EXTRACT NAME AND STATE FROM THE PROFILE 
def extract_profile(text, idx):
    name_match = re.search(r"Name of respondent\s+(.*)", text, re.I)
    name = name_match.group(1).strip() if name_match else f"Group_{idx}"

    state_match = re.search(r"State\s+(.*)", text, re.I)
    state = state_match.group(1).strip() if state_match else "Unknown"

    return name,state


In [16]:
# DIVIDE EACH SESSION INTO SECTIONS BY HEADERS AND STORE THE CONTENT SESIONWISE
SECTION_HEADERS = [
    "Business establishment",
    "Managing business activities",
    "Risks or challenges faced and perceived in business. Ways to handle them.",
    "Access and usage levels of banking products",
    "Access and usage levels of digital infrastructure",
    "Past learning experience and training support required",
    "Willingness to share personal and business data"
]

def extract_sections(text):
    sections = {}

    # normalize
    text = re.sub(r'\s+', ' ', text)

    # find headings
    pattern = "|".join([re.escape(h) for h in SECTION_HEADERS])
    matches = list(re.finditer(pattern, text, re.I))

    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)

        heading = match.group(0).strip()
        content = text[start:end].strip()

        sections[heading] = content

    return sections



In [17]:
#PROCESSING PIPLEINE FOR EACH PDF FILE
rows=[]
def run_pipeline(pdf_path):
    print(f"Reading File:{pdf_path}")
    text = read_pdf(pdf_path)

    print("Splitting sessions")
    sessions = split_sessions(text)


    for i, session in enumerate(sessions):
        row = {}

        # ID + Name
        row["Respondent ID"] = f"rid_{i+1}"
        row["Name of Respondent"],row["State"] = extract_profile(session, i+1)

        # Sections
        sections = extract_sections(session)
        print(sections)

        for key, value in sections.items():
            row[key] = value

        rows.append(row)


    print("Completed")
    return


In [18]:
def main():
    for i in range (1,5):
        run_pipeline(f"doc{i}.pdf")
#     run_pipeline("doc1.pdf")
#     run_pipeline("doc2.pdf")
#     run_pipeline("doc3.pdf")
#     run_pipeline("doc4.pdf")

    df = pd.DataFrame(rows)
    print(df.head())

    df.to_excel("structured_output.xlsx", index=False)


In [19]:
if __name__ == "__main__":
    main()

Reading File:doc1.pdf
Splitting sessions
{'Business establishment': "The idea of becoming a weaver/job worker for Seva Mandir's business enterprise 'Sadhna' was introduced to them by Seva Mandir's staff. Two of them used to work in a nearby chemical factory (closed few years back) and the other two were housewives. No one in the group had any experience of sewing cloths or weaving for commercial purposes. Two members attended the formal training sessions that were conducted by Seva Mandir and rest of the members of the group learnt by working along with the trained members. The members who got training were trained for 3 months (10 am- 5 pm) on stitching, embroidery and patch work. These were classroom based training at Seva Mandir’s office. The group operates as an individual unit and works directly for Sadhna. The monthly turnover for the entrepreneurs is INR 8000-10,000 per month. On an average - major expenses categories are house rent, children education, electricity bill (INR 150

Splitting sessions
{'Business establishment': 'and', 'managing business activities': 'All the women who were part of the group have been supporting varied family business mainly managed by the entire household. Therefore, they have been playing the role of an unpaid employee who have no decision- making power in the business activities or specifically business finances. Most women have their family businesses located next to their houses which makes it convenient for them to manage the shops. While most of the women received support from their children and in-laws on the business in managing the shop, their husbands were involved in other income opportunities e.g. contractual jobs, part-time jobs, other businesses etc. However, these women feel that although they are fully occupied with the business and household responsibilities, they can still spare a few hours every week for an independent business opportunity. The intention of starting such a business is to specifically have the in

Splitting sessions
{'Business establishment': 'His father established the bakery shop 35 years ago in 1984 and which has become one of the busiest markets of Kolkata now - Hathibagan. Then in 2001, their father thought ahead and decided to expand into manufacturing bakery items with the clear intention that within this field it is difficult to control quality of the final product if the quality of raw material is not adequate. Their father decided to start a small cow shed to help create basic raw material i.e. milk for their products. However, the surplus milk was either supplied to customers directly or converted into ghee to be sold through their shop. This way, their father not only tried to maintain the quality of products their bakery created while also removing dependency on other suppliers and creating multiple business streams to reduce volatility risk for their business. This practice still continues.', 'Managing business activities': 'He has a very loyal customer base (end-c

Splitting sessions
{'Business establishment': 'Mr. K Sainath has two running shops in his name. The first shop is 7 years old and the second shop is 4 years old. Sainath started his business with the photocopy and cybercafé services, 7 years ago at the age of 21. He continued this business for some years but after seeing decline in footfall for cyber café needs, Mr. Sainath diversified his business and became an authorized service provider of online services under Mee Seva (Integrated Service Delivery Gateway of Telangana). This has diversified his source of income. Later on, Mr. Sainath also opened another shop (the second shop). The second shop is also into photocopy services and provide online TIN Facilitation services authorized by NSDL. This shop is near college so have high demand of photocopy as well as other form filling requirements. Both the shops together accounts monthly sales of Rs. 120,000 to 160,000.', 'Managing business activities': 'It took Mr. Sainath, INR 250,000 ini